# scCODA compositional analysis — `annot_5` vs doctor-defined clinical outcome

Patient is the unit of analysis. 24 patients (13 Good / 11 Poor), 26 `annot_5` populations,
Good as baseline, so a positive coefficient means **relatively enriched in Poor**.

**Sections**

| # | Section | Runtime | Default |
|---|---|---|---|
| 1 | Setup and configuration | instant | on |
| 2 | Load metadata and validate | ~20 s | on |
| 3 | Count matrix and sparsity | instant | on |
| 4 | **Main scCODA fit** | ~3.5 min | on |
| 5 | Results table | instant | on |
| 6 | Proportions + **figures 1 and 2** | instant | on |
| 7 | Reference sensitivity (3 extra fits) | ~10 min | **off** — done, review doc 5.1 |
| 8 | Seed stability (2 extra fits) | ~7 min | **off** — done, review doc 5.2/5.3 |
| 9 | Leave-one-site-out (9 fits, 20k draws) | ~9 min | **off** — done, review doc 5.4 |
| 10 | Marker-based lineage check | ~2 min | on |
| 11 | Share index + **figure 3** | instant | on |
| 12 | CSV exports | instant | off |

**As configured this run takes about 6 minutes.** Sections 7-9 are the expensive robustness
refits; they were already completed at 100,000 draws and are recorded in
`scCODA_critical_review_and_final_plan.md` sections 5.1-5.4, so they are off by default. Set
the matching `RUN_*` flag in section 1 back to `True` to reproduce any of them (~45 min for all).

**Figures.** Three SVGs are written to `outputs/images/` — the same folder `sc_coda_initial_attempt.ipynb` uses:

| file | content |
|---|---|
| `sc_coda_boxplot_credible_annot_5.svg` | the credible populations, Good vs Poor |
| `sc_coda_cell_type_distribution_box_whisker_annot_5.svg` | all 26 populations — the `annot_5` counterpart to `sc_coda_initial_attempt.ipynb`'s `..._annot_6.svg` |
| `sc_coda_myoepithelial_share_annot_5.svg` | myoepithelial/basal share + within-lineage cycling rate |

**Writing.** Two independent switches in section 1: `SAVE_FIGURES` (on — SVGs only) and
`SAVE_OUTPUTS` (off — CSVs). The `.h5ad` is opened read-only and is never written under either.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


## 1. Setup and configuration

In [ ]:
import os
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # quieten TensorFlow startup chatter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import anndata as ad
import tensorflow as tf
from IPython.display import display
from scipy.stats import mannwhitneyu
from sccoda.util import cell_composition_data as dat
from sccoda.util import comp_ana as mod

# ---------------------------------------------------------------- paths
H5AD_PATH      = Path(f"{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad")
CLINICAL_INFO  = Path(f"{ACC_DATA_ROOT}/clinical_info.csv")
OUTPUT_DIR     = Path(f"{ACC_DATA_ROOT}/outputs/csv_files/scCODA/annot_5_final")
IMAGE_DIR      = Path(f"{ACC_DATA_ROOT}/outputs/images")   # same folder sc_coda_initial_attempt.ipynb uses
FIG_PREFIX     = "sc_coda"             # matches sc_coda_initial_attempt.ipynb's existing annot_6 figure

# ---------------------------------------------------------------- design
SAMPLE_COL       = "batch"             # one row per patient
CELL_TYPE_COL    = "annot_5"           # 26 populations
OUTCOME_COL      = "clinical_outcome"  # doctor-defined; Good vs Poor
BASELINE_LEVEL   = "Good"              # so positive coefficient = enriched in Poor
COMPARISON_LEVEL = "Poor"
REFERENCE_CELL_TYPE = "automatic"      # scCODA picks; we validate the choice in section 4

# ---------------------------------------------------------------- sampling
SEED          = 20260729
NUM_RESULTS   = 100_000                # posterior draws
NUM_BURNIN    = 10_000
ESTIMATED_FDR = 0.05

# ---------------------------------------------------------------- optional checks
# 7, 8 and 9 are already finished and recorded in scCODA_critical_review_and_final_plan.md
# (sections 5.1-5.4) from 11 refits at 100k draws. They are off here so a figure run takes
# ~6 min instead of ~45. Set any back to True to reproduce them.
RUN_REFERENCE_SENSITIVITY = False  # ~10 min  (section 7)  - done, see review doc 5.1
RUN_SEED_STABILITY        = False  # ~7 min   (section 8)  - done, see review doc 5.2/5.3
RUN_LEAVE_SITE_OUT        = False  # ~9 min   (section 9)  - done, see review doc 5.4
RUN_MARKER_CHECK          = True   # ~2 min   (section 10) - validates the lineage buckets
RUN_SHARE_INDEX           = True   # instant  (section 11) - feeds the headline figure

SWEEP_DRAWS  = 20_000    # shorter chains for the 9-fit site sweep
SWEEP_BURNIN = 5_000

# ---------------------------------------------------------------- safety
# Two separate switches. Figures are safe to write and are what this run is for;
# CSV exports stay off by default. The .h5ad is never written under either.
SAVE_FIGURES = True      # SVGs -> IMAGE_DIR
SAVE_OUTPUTS = False     # CSVs -> OUTPUT_DIR

def save_fig(fig, name):
    """Write an SVG to IMAGE_DIR when SAVE_FIGURES is on. Returns the path, or None."""
    if not SAVE_FIGURES:
        print(f"  [SAVE_FIGURES = False] not written: {FIG_PREFIX}_{name}.svg")
        return None
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    path = IMAGE_DIR / f"{FIG_PREFIX}_{name}.svg"
    fig.savefig(path, format="svg", bbox_inches="tight")
    print(f"  figure written: {path}")
    return path

print(f"annotation : {CELL_TYPE_COL}")
print(f"outcome    : {OUTCOME_COL}  ({COMPARISON_LEVEL} vs {BASELINE_LEVEL})")
print(f"sampling   : {NUM_RESULTS:,} draws, {NUM_BURNIN:,} burn-in, seed {SEED}")
print(f"figures    : {SAVE_FIGURES}  -> {IMAGE_DIR}")
print(f"csv exports: {SAVE_OUTPUTS}")

## 2. Load metadata and validate

Only `adata.obs` is needed for the compositional model, so the file is opened in backed mode
and the 8 GB expression matrix is never read into RAM here.

`site` in the h5ad has two defects: `BOT` and `BaseofTongue` are the same anatomical site
recorded under two labels, and `"NA"` is a literal string rather than missing data. Site is
therefore rebuilt from `clinical_info.csv`, which is the authoritative source. This only
affects the section 9 sweep, not the main result.

In [ ]:
adata = ad.read_h5ad(H5AD_PATH, backed="r")
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes")

obs = adata.obs[[SAMPLE_COL, CELL_TYPE_COL, OUTCOME_COL]].copy()

# --- fail loudly rather than silently analysing broken metadata -------------
if obs.isna().any().any():
    raise ValueError(f"NaNs in required columns:\n{obs.isna().sum()}")

levels_found = set(obs[OUTCOME_COL].astype(str).unique())
if levels_found != {BASELINE_LEVEL, COMPARISON_LEVEL}:
    raise ValueError(f"expected {{{BASELINE_LEVEL}, {COMPARISON_LEVEL}}}, found {levels_found}")

# one outcome per patient, no exceptions
per_patient_levels = obs.groupby(SAMPLE_COL, observed=True)[OUTCOME_COL].nunique()
if (per_patient_levels > 1).any():
    raise ValueError(f"patients with >1 outcome label: {per_patient_levels[per_patient_levels>1].index.tolist()}")

# explicit category order makes Good the treatment-coding baseline
obs[OUTCOME_COL] = pd.Categorical(obs[OUTCOME_COL].astype(str),
                                  categories=[BASELINE_LEVEL, COMPARISON_LEVEL], ordered=True)

patient_meta = obs.groupby(SAMPLE_COL, observed=True)[[OUTCOME_COL]].first()

# --- authoritative anatomical site ----------------------------------------
clin = pd.read_csv(CLINICAL_INFO)
clin.columns = [c.strip().lstrip("\ufeff") for c in clin.columns]
site_map = clin.set_index("Sample ID")["Site of Extraction"]
patient_meta["site"] = patient_meta.index.astype(str).map(site_map)
if patient_meta["site"].isna().any():
    missing = patient_meta.index[patient_meta["site"].isna()].tolist()
    raise ValueError(f"no site in clinical_info.csv for: {missing}")

print("\nclinical outcome:")
print(patient_meta[OUTCOME_COL].value_counts(sort=False).to_string())
print("\nsite x outcome (site rebuilt from clinical_info.csv):")
display(pd.crosstab(patient_meta["site"], patient_meta[OUTCOME_COL], margins=True))

## 3. Count matrix and sparsity

Cells are aggregated into a patient-by-population count matrix, so scCODA compares 13 patients
with 11 patients rather than treating 95,441 cells as independent observations.

The all-zero column guard is precautionary. No population is empty across the full cohort, but
a leave-out analysis could in principle empty one, and scCODA would then silently fit a column
consisting entirely of its 0.5 pseudocount.

In [ ]:
def build_counts(obs_df, keep_patients=None):
    """Patient x population count matrix, dropping any population that is empty everywhere."""
    o = obs_df if keep_patients is None else obs_df[obs_df[SAMPLE_COL].isin(keep_patients)]
    counts = pd.crosstab(o[SAMPLE_COL], o[CELL_TYPE_COL])
    dropped = counts.columns[counts.sum(axis=0) == 0].tolist()
    if dropped:
        print(f"  dropped {len(dropped)} all-zero population(s): {dropped}")
    return counts.loc[:, counts.sum(axis=0) > 0]

counts = build_counts(obs)
patient_meta = patient_meta.reindex(counts.index)
print(f"count matrix: {counts.shape[0]} patients x {counts.shape[1]} populations")
print(f"cells per patient: min {counts.sum(1).min():,}  median {counts.sum(1).median():,.0f}  max {counts.sum(1).max():,}")

# Computed for the CSV export in section 12, not displayed - it is a QC artefact, not a
# result, and 26 rows of it on screen buries the two numbers above that actually matter.
sparsity = pd.DataFrame({
    "total_cells":      counts.sum(axis=0),
    "patients_with_zero": (counts == 0).sum(axis=0),
    "min_per_patient":  counts.min(axis=0),
    "median_per_patient": counts.median(axis=0),
}).sort_values("patients_with_zero", ascending=False)
print(f"sparsity table computed ({(sparsity['patients_with_zero'] >= 10).sum()} of "
      f"{len(sparsity)} populations absent from 10+ patients); exported by section 12.")

print("\nscCODA adds a 0.5 pseudocount where zeros occur. Sparse populations cannot be tested")
print("meaningfully, but they do not affect which populations are selected (see section 5).")
print("Do not add a prevalence filter after looking at significance.")

## 4. Main scCODA fit

Seeds are set immediately before model construction and sampling, so re-running this cell alone
reproduces the fit.

scCODA's automatic reference selection minimises `var/mean` of relative abundance among
populations present in >95% of patients. That criterion scales with the mean and so tends to
pick *rare* populations; section 7 refits with three abundant alternatives to confirm the
result does not depend on the choice. Note that the reference has its coefficient fixed at zero
by construction, so it can never be reported as credible.

In [ ]:
def fit_sccoda(count_df, meta_df, reference=REFERENCE_CELL_TYPE, seed=SEED,
               num_results=NUM_RESULTS, num_burnin=NUM_BURNIN, verbose=True):
    """Fit scCODA and return (model, result, reference_population_name)."""
    frame = count_df.join(meta_df[[OUTCOME_COL]])
    data  = dat.from_pandas(frame, covariate_columns=[OUTCOME_COL])

    np.random.seed(seed)
    tf.random.set_seed(seed)

    model = mod.CompositionalAnalysis(data, formula=OUTCOME_COL, reference_cell_type=reference)
    ref_name = str(data.var_names[int(model.reference_cell_type)])
    result = model.sample_hmc(num_results=num_results, num_burnin=num_burnin, verbose=verbose)
    return model, result, ref_name

model, result, ref_name = fit_sccoda(counts, patient_meta)

print(f"\nreference population : {ref_name}")
print(f"acceptance rate      : {float(result.sample_stats['is_accepted'].mean()):.1%}")

# is the reference actually stable between outcome groups?
ref_prop = counts[ref_name] / counts.sum(axis=1)
g = ref_prop[patient_meta[OUTCOME_COL] == BASELINE_LEVEL]
p = ref_prop[patient_meta[OUTCOME_COL] == COMPARISON_LEVEL]
print(f"reference present in all {int((counts[ref_name] > 0).sum())}/{len(counts)} patients, "
      f"min {int(counts[ref_name].min())} cells")
print(f"reference proportion Good {g.median():.3%} vs Poor {p.median():.3%}, "
      f"Mann-Whitney p = {mannwhitneyu(p, g).pvalue:.3f}")

## 5. Results

Two columns that are easy to confuse:

- **`inclusion_probability`** — the share of posterior draws in which the effect was non-zero.
  This is the actual evidence, and it is never thresholded.
- **`fdr_thresholded_coefficient` / `..._log2_fold_change`** — scCODA sets these to exactly 0
  when the effect does not clear the selection threshold. **A zero here does not mean the
  posterior effect was zero.** This is why the original notebook's table appeared to be all
  zeros.

How the threshold is chosen matters for reading the output. scCODA picks the lowest inclusion
probability `c` such that the mean of `(1 - inclusion probability)` over all effects at or above
`c` stays under 5%. Two consequences:

- If only one effect is a candidate, the rule reduces to `inclusion probability > 0.95`.
- With several strong effects the bar for the weakest is *lower*. Here, given two effects near
  1.0, the third is admitted down to roughly 0.85.

So the threshold is a joint property of the fit, not a fixed 0.95 per population. Sparse
populations sit near 0.5 and never enter the calculation.

In [ ]:
def results_table(res, est_fdr=ESTIMATED_FDR):
    """Posterior evidence and FDR-thresholded effects, with the MultiIndex kept as columns."""
    _, eff = res.summary_prepare(est_fdr=est_fdr)
    eff = eff.copy()

    out = eff.reset_index().rename(columns={
        "Inclusion probability": "inclusion_probability",
        "Final Parameter":       "fdr_thresholded_coefficient",
        "log2-fold change":      "fdr_thresholded_log2_fold_change",
        "HDI 3%":  "hdi_3", "HDI 97%": "hdi_97", "SD": "posterior_sd",
    })

    # credibility from the threshold itself, not from a coefficient that happens to be zero
    threshold = res.model_specs["threshold_prob"]
    out["credible"] = out["inclusion_probability"] >= threshold

    out["fold_change_vs_" + BASELINE_LEVEL] = np.power(2.0, out["fdr_thresholded_log2_fold_change"])
    out["direction"] = "not credible"
    out.loc[out["credible"] & (out["fdr_thresholded_log2_fold_change"] > 0), "direction"] = f"enriched in {COMPARISON_LEVEL}"
    out.loc[out["credible"] & (out["fdr_thresholded_log2_fold_change"] < 0), "direction"] = f"enriched in {BASELINE_LEVEL}"

    return out.sort_values("inclusion_probability", ascending=False).reset_index(drop=True), threshold

results, threshold = results_table(result)

# guard: confirm we really are looking at Poor vs Good and not the reverse
expected = f"{OUTCOME_COL}[T.{COMPARISON_LEVEL}]"
found = set(results["Covariate"].astype(str))
if found != {expected}:
    raise RuntimeError(f"expected coefficient {expected!r}, got {found}. Do not read directions yet.")

credible = results[results["credible"]]
print(f"coefficient          : {expected}  (positive = enriched in {COMPARISON_LEVEL})")
print(f"selected threshold   : {threshold:.3f}   (target estimated FDR {ESTIMATED_FDR:.0%})")
print(f"credible populations : {len(credible)} of {len(results)}\n")

display(results[["Cell Type", "inclusion_probability",
                 "fdr_thresholded_log2_fold_change", f"fold_change_vs_{BASELINE_LEVEL}",
                 "hdi_3", "hdi_97", "posterior_sd", "credible", "direction"]])

## 6. Patient-level proportions

scCODA works on the full count composition; these proportions are the raw descriptive picture
behind it. The Mann-Whitney p-values are computed only for populations the model already
selected, so they are **post-selection** and biased low — they are a sanity check on direction,
not independent evidence.

In [ ]:
props = counts.div(counts.sum(axis=1), axis=0)
props_out = props.join(patient_meta[[OUTCOME_COL]])

to_show = credible["Cell Type"].tolist() or results.head(5)["Cell Type"].tolist()

rows = []
for ct in to_show:
    g = props_out.loc[props_out[OUTCOME_COL] == BASELINE_LEVEL, ct]
    p = props_out.loc[props_out[OUTCOME_COL] == COMPARISON_LEVEL, ct]
    rows.append({
        "Cell Type": ct,
        f"{BASELINE_LEVEL}_median": f"{g.median():.1%}", f"{BASELINE_LEVEL}_mean": f"{g.mean():.1%}",
        f"{COMPARISON_LEVEL}_median": f"{p.median():.1%}", f"{COMPARISON_LEVEL}_mean": f"{p.mean():.1%}",
        "post_selection_p_not_valid_inference": round(mannwhitneyu(p, g).pvalue, 4),
    })
proportion_summary = pd.DataFrame(rows)
display(proportion_summary)

# ------------------------------------------------------------------ plotting
# One patient = one point. Boxes are the group distribution, points are the patients,
# so a reader can count the n and see the spread rather than trusting the box.
#
# Palette is the project's paper convention: green = Good outcome, red = Poor outcome.
# Source of truth is bulk_rna_data/scripts/named_scripts/wgcna_final.R, which draws the
# same kind of figure (boxplot by clinical outcome + jittered stripchart) with
#     col = c("#4DAF4A", "#E41A1C")   levels = c("5Plus", "2Minus")
#     stripchart(..., pch = 19, col = "black")
# Do not re-derive these; change them there and here together.
GROUP_COLOURS = {BASELINE_LEVEL: "#4DAF4A", COMPARISON_LEVEL: "#E41A1C"}
POINT_COLOUR  = "black"       # matches the stripchart in wgcna_final.R
BOX_EDGE      = "#333333"
WHISKER_STYLE = (0, (5, 3))   # dashed whiskers, as base-R boxplot() draws them by default,
                              # so these match the other box plots in the paper.
                              # Caps and medians stay solid.

def boxplot_panel(cell_types, ncols=3, annotate_p=True, title=None, panel_w=4.2, panel_h=3.6):
    """Grid of Good-vs-Poor proportion box plots, one subplot per population."""
    rng   = np.random.default_rng(SEED)          # reproducible jitter
    n     = len(cell_types)
    ncols = min(ncols, max(1, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w*ncols, panel_h*nrows), squeeze=False)

    for i, ct in enumerate(cell_types):
        ax = axes[i // ncols, i % ncols]
        vals = []
        for lvl in [BASELINE_LEVEL, COMPARISON_LEVEL]:
            vals.append(props_out.loc[props_out[OUTCOME_COL] == lvl, ct].to_numpy() * 100)

        # boxes first (filled, group colour), points on top in black
        bp = ax.boxplot(vals, labels=[f"{BASELINE_LEVEL}\n(n={vals[0].size})",
                                      f"{COMPARISON_LEVEL}\n(n={vals[1].size})"],
                        showfliers=False, widths=.55, zorder=2, patch_artist=True)
        for box, lvl in zip(bp["boxes"], [BASELINE_LEVEL, COMPARISON_LEVEL]):
            box.set_facecolor(GROUP_COLOURS[lvl])
            box.set_edgecolor(BOX_EDGE)
            box.set_linewidth(1.2)
        for element in ("whiskers", "caps", "medians"):
            for artist in bp[element]:
                artist.set_color(BOX_EDGE)
                artist.set_linewidth(1.2)
        for w in bp["whiskers"]:            # dashed whiskers only; caps/medians stay solid
            w.set_linestyle(WHISKER_STYLE)

        for j, v in enumerate(vals, start=1):
            ax.scatter(j + rng.uniform(-.10, .10, v.size), v, s=30, alpha=.9, zorder=3,
                       color=POINT_COLOUR, linewidths=0)

        if annotate_p:
            pv = mannwhitneyu(vals[1], vals[0]).pvalue
            ax.set_title(f"{ct}\np = {pv:.3g}", fontsize=9)
        else:
            ax.set_title(ct, fontsize=9)
        ax.set_ylabel("% of cells", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    for j in range(n, nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")
    if title:
        fig.suptitle(title, fontsize=12, y=1.005)
    fig.tight_layout()
    return fig

# --- main figure: the populations the model selected -----------------------
fig_credible = boxplot_panel(to_show, ncols=3,
                             title=f"scCODA credible populations ({CELL_TYPE_COL}) — "
                                   f"{COMPARISON_LEVEL} vs {BASELINE_LEVEL}")
save_fig(fig_credible, f"boxplot_credible_{CELL_TYPE_COL}")
plt.show()

# --- supplementary: every population, ordered by inclusion probability -----
# This is the figure that shows the immune/stromal compartment is flat. Ordering by
# inclusion probability puts the credible populations first and the null ones after,
# so "nothing else moved" is visible rather than asserted.
all_ordered = [ct for ct in results["Cell Type"] if ct in props_out.columns]
fig_all = boxplot_panel(all_ordered, ncols=4, annotate_p=False,
                        title=f"All {len(all_ordered)} {CELL_TYPE_COL} populations, "
                              f"ordered by posterior inclusion probability",
                        panel_w=3.4, panel_h=3.0)
# direct counterpart to sc_coda_initial_attempt.ipynb's sc_coda_cell_type_distribution_box_whisker_annot_6.svg
save_fig(fig_all, f"cell_type_distribution_box_whisker_{CELL_TYPE_COL}")
plt.show()

print("\nNote: p-values on the credible panel are post-selection (the populations were chosen")
print("by the model on the same data) and are descriptive only. The all-population figure")
print("omits them deliberately.")

## 7. Reference sensitivity  *(optional, ~10 min)*

The reference population is fixed at zero and every other effect is measured relative to it, so
it is worth confirming the answer is not an artefact of that one choice. Three abundant
alternatives are refitted here. If the credible set and the fold changes agree, the reference
question is settled and can be dismissed.

In [ ]:
if RUN_REFERENCE_SENSITIVITY:
    alternatives = [c for c in ["Endothelial cells", "Macrophage - M2", "Fibroblast cells"]
                    if c in counts.columns]
    rows = [{"reference": ref_name, "threshold": round(threshold, 3),
             **{ct: round(v, 4) for ct, v in zip(results["Cell Type"], results["inclusion_probability"]) if ct in to_show},
             "credible_set": ", ".join(sorted(credible["Cell Type"]))}]

    for alt in alternatives:
        print(f"\n--- refitting with reference = {alt} ---")
        _, res_alt, name_alt = fit_sccoda(counts, patient_meta, reference=alt, verbose=False)
        tab_alt, thr_alt = results_table(res_alt)
        cred_alt = tab_alt[tab_alt["credible"]]
        rows.append({"reference": name_alt, "threshold": round(thr_alt, 3),
                     **{ct: round(float(tab_alt.loc[tab_alt["Cell Type"] == ct, "inclusion_probability"].iloc[0]), 4)
                        for ct in to_show},
                     "credible_set": ", ".join(sorted(cred_alt["Cell Type"]))})

    reference_sensitivity = pd.DataFrame(rows)
    print("\nInclusion probability by reference population:")
    display(reference_sensitivity)
    n_sets = reference_sensitivity["credible_set"].nunique()
    print("IDENTICAL credible set across all references." if n_sets == 1
          else f"WARNING: {n_sets} different credible sets - the result depends on the reference.")
else:
    print("skipped (RUN_REFERENCE_SENSITIVITY = False)")

## 8. Seed stability  *(optional, ~7 min)*

The credible/not-credible call is a hard cut on a Monte Carlo estimate, so an effect sitting
near the threshold can flip between runs purely from sampling noise. Two extra seeds are fitted
and a batch-means Monte Carlo standard error is reported, which accounts for autocorrelation
between draws. If an inclusion probability sits within about one MCSE of the threshold, the
decision should be described as marginal rather than robust.

In [ ]:
def mcse_inclusion(res, n_batches=50):
    """Batch-means Monte Carlo SE of each inclusion probability."""
    beta = np.asarray(res.posterior["beta"])[0]
    m = beta.shape[0] // n_batches
    out = {}
    for k, ct in enumerate(res.posterior.coords["cell_type"].values):
        ind = (np.abs(beta[:, 0, k]) > 1e-3).astype(float)
        bm = np.array([ind[i*m:(i+1)*m].mean() for i in range(n_batches)])
        out[str(ct)] = bm.std(ddof=1) / np.sqrt(n_batches)
    return out

if RUN_SEED_STABILITY:
    se0 = mcse_inclusion(result)
    rows = [{"seed": SEED, "threshold": round(threshold, 3),
             **{ct: f"{float(results.loc[results['Cell Type']==ct,'inclusion_probability'].iloc[0]):.4f} "
                    f"(+-{se0[ct]:.4f})" for ct in to_show},
             "n_credible": int(len(credible))}]

    for s in [1, 2]:
        print(f"\n--- refitting with seed = {s} ---")
        _, res_s, _ = fit_sccoda(counts, patient_meta, seed=s, verbose=False)
        tab_s, thr_s = results_table(res_s)
        se_s = mcse_inclusion(res_s)
        rows.append({"seed": s, "threshold": round(thr_s, 3),
                     **{ct: f"{float(tab_s.loc[tab_s['Cell Type']==ct,'inclusion_probability'].iloc[0]):.4f} "
                            f"(+-{se_s[ct]:.4f})" for ct in to_show},
                     "n_credible": int(tab_s["credible"].sum())})

    seed_stability = pd.DataFrame(rows)
    display(seed_stability)
    print("Same number of credible populations across seeds." if seed_stability["n_credible"].nunique() == 1
          else "WARNING: the credible set changes between seeds - report the decision as unstable.")
else:
    print("skipped (RUN_SEED_STABILITY = False)")

## 9. Leave-one-site-out  *(optional, ~9 min)*

This does **not** question any outcome label. The labels are clinical fact.

What it asks is narrower: our claim is that a composition is associated with outcome, but
anatomical site is not evenly spread across the outcome groups. If dropping one site removes
the result, then site and composition cannot be told apart in this cohort, and that belongs in
the limitations. Reviewers ask this; it is cheaper to have the answer.

Shorter chains (20,000 draws) are used because this is 9 refits. Borderline calls are noisier
at that length, so read the inclusion probabilities rather than the binary verdict.

The reference population is pinned to the one chosen in section 4. Left on `"automatic"`, the
selection can land on a different population in each subset, and the fits would no longer be
comparable to each other.

**Known result at full length.** Removing the four Maxilla patients was checked separately at
100,000 draws: the credible set became **empty** at both `annot_5` and `annot_6`. Inclusion
probabilities fell to Myoepithelial 0.890, Epithelial 0.873, Actively Dividing 0.572. If the
20,000-draw sweep below disagrees, trust the longer run. To confirm, set `SWEEP_DRAWS = 100_000`
(about 30 minutes for the full sweep).

In [ ]:
if RUN_LEAVE_SITE_OUT:
    rows = []
    for site in sorted(patient_meta["site"].unique()):
        dropped = patient_meta.index[patient_meta["site"] == site]
        keep    = patient_meta.index.difference(dropped)
        meta_k  = patient_meta.loc[keep]
        n_g = int((meta_k[OUTCOME_COL] == BASELINE_LEVEL).sum())
        n_p = int((meta_k[OUTCOME_COL] == COMPARISON_LEVEL).sum())
        if n_g < 3 or n_p < 3:
            print(f"skipping {site}: only {n_g} Good / {n_p} Poor would remain")
            continue

        counts_k = build_counts(obs, keep_patients=keep)
        # pin the reference to the main fit's, otherwise automatic selection can pick a
        # different population in each subset and the fits stop being comparable
        _, res_k, _ = fit_sccoda(counts_k, meta_k, reference=ref_name, seed=SEED,
                                 num_results=SWEEP_DRAWS, num_burnin=SWEEP_BURNIN, verbose=False)
        tab_k, thr_k = results_table(res_k)
        cred_k = tab_k[tab_k["credible"]]["Cell Type"].tolist()
        rows.append({
            "site_removed": site, "n_removed": len(dropped),
            "remaining": f"{n_g}G/{n_p}P", "threshold": round(thr_k, 3),
            **{ct: round(float(tab_k.loc[tab_k["Cell Type"] == ct, "inclusion_probability"].iloc[0]), 3)
               for ct in to_show if ct in set(tab_k["Cell Type"])},
            "n_credible": len(cred_k),
        })
        print(f"  removed {site:22s} ({len(dropped)} patients) -> {len(cred_k)} credible")

    leave_site_out = pd.DataFrame(rows)
    display(leave_site_out)
    fragile = leave_site_out[leave_site_out["n_credible"] == 0]["site_removed"].tolist()
    if fragile:
        print(f"\nCredible set empties when these sites are removed: {fragile}")
        print("Report this. Note it is a harsh test: removing a site also removes patients,")
        print("so it mixes confounding with plain loss of statistical power.")
    else:
        print("\nCredible set survives removal of every individual site.")
else:
    print("skipped (RUN_LEAVE_SITE_OUT = False)")

## 10. Marker-based lineage check  *(optional, ~2 min)*

This is not a criticism of the annotation — it validates it, and it guards against one specific
mistake in the write-up.

Two things to know. First, `adata.X` is z-scaled, so cell-cycle genes look flat there; the raw
`layers['counts']` must be used. Second, three populations have "Epithelial" in their name but
are myoepithelial/basal by marker profile, so grouping by name string puts them in the wrong
bucket.

What the panels answer: is `Actively Dividing cells` its own lineage, or is it the cycling
fraction of a lineage that is already in the table? If the latter, then "more proliferation in
Poor" is the wrong sentence to write.

In [ ]:
if RUN_MARKER_CHECK:
    import h5py

    MYO_MARKERS = ["KRT14", "ACTA2", "TP63", "MYH11"]           # myoepithelial / basal
    LUM_MARKERS = ["KIT", "KRT7", "KRT8", "EPCAM", "CLDN4"]     # luminal / ductal
    PRO_MARKERS = ["MKI67", "TOP2A"]                            # proliferation
    panel = [g for g in MYO_MARKERS + LUM_MARKERS + PRO_MARKERS if g in adata.var_names]

    var_names = list(adata.var_names)
    idx = np.array([var_names.index(g) for g in panel])
    lookup = -np.ones(adata.n_vars, dtype=np.int64)
    lookup[idx] = np.arange(len(idx))

    # stream the CSR counts layer in row blocks and pick out just these columns
    with h5py.File(H5AD_PATH, "r") as f:
        layer = f["layers/counts"]
        data, indices, indptr = layer["data"], layer["indices"], layer["indptr"][:]
        mat = np.zeros((adata.n_obs, len(panel)), dtype=np.float32)
        for start in range(0, adata.n_obs, 20_000):
            stop = min(start + 20_000, adata.n_obs)
            lo, hi = int(indptr[start]), int(indptr[stop])
            d, ix = data[lo:hi], indices[lo:hi]
            rows = np.repeat(np.arange(start, stop), np.diff(indptr[start:stop+1]).astype(int))
            k = lookup[ix]
            keep = k >= 0
            np.add.at(mat, (rows[keep], k[keep]), d[keep])

    # normalise to counts per 10k UMI so populations are comparable
    per10k = mat / adata.obs["total_counts"].to_numpy(dtype=float)[:, None] * 1e4
    expr = pd.DataFrame(per10k, columns=panel)
    expr["population"] = adata.obs[CELL_TYPE_COL].to_numpy()

    tumour = [c for c in counts.columns if "Tumor" in c or c == "Actively Dividing cells"]
    means = expr[expr["population"].isin(tumour)].groupby("population", observed=True)[panel].mean()

    lineage = pd.DataFrame({
        "myoepithelial_panel": means[[g for g in MYO_MARKERS if g in panel]].mean(axis=1),
        "luminal_panel":       means[[g for g in LUM_MARKERS if g in panel]].mean(axis=1),
        "proliferation_panel": means[[g for g in PRO_MARKERS if g in panel]].mean(axis=1),
    })
    lineage["myo_over_lum"] = lineage["myoepithelial_panel"] / lineage["luminal_panel"]
    lineage["assignment"] = np.where(lineage["myo_over_lum"] > 1, "myoepithelial/basal", "luminal/ductal")
    display(lineage.sort_values("myo_over_lum", ascending=False).round(2))

    # detection rate confirms the proliferation label is real, not an artefact
    det = (expr[expr["population"].isin(tumour)]
           .assign(**{g: expr[g] > 0 for g in PRO_MARKERS if g in panel})
           .groupby("population", observed=True)[[g for g in PRO_MARKERS if g in panel]].mean() * 100)
    print("\n% of cells with at least one count (proliferation markers):")
    display(det.round(1))
else:
    print("skipped (RUN_MARKER_CHECK = False)")

## 11. Myoepithelial/basal share of the tumour compartment  *(optional, instant)*

A single summary of the same biology: of the tumour cells only, what fraction is
myoepithelial/basal? Immune, stromal and normal-epithelial cells are excluded, so the number is
unaffected by how much non-tumour tissue happened to be in the dissociated piece.

This is offered as a compact way to describe the result, not as a replacement for scCODA. It
was derived after seeing the model output, so it is a re-expression of these data rather than a
pre-registered endpoint; it would need an independent cohort to be treated as one.

In [ ]:
if RUN_SHARE_INDEX:
    MYO_BUCKET = [c for c in ["Myoepithelial cells - Tumor",
                              "Actively Dividing Myoepithelial cells - Tumor",
                              "Epithelial cells - Basal - Tumor",
                              "Basal cells - Tumor"] if c in counts.columns]
    LUM_BUCKET = [c for c in ["Epithelial cells - Tumor",
                              "Actively Dividing cells"] if c in counts.columns]
    print(f"myoepithelial/basal : {MYO_BUCKET}")
    print(f"luminal/ductal      : {LUM_BUCKET}\n")

    tumour_total = counts[MYO_BUCKET + LUM_BUCKET].sum(axis=1)
    share = counts[MYO_BUCKET].sum(axis=1) / tumour_total

    share_table = pd.DataFrame({
        "myoepithelial_basal_share": share.round(3),
        "tumour_cells": tumour_total,
        "tumour_purity": (tumour_total / counts.sum(axis=1)).round(3),
    }).join(patient_meta)
    display(share_table.sort_values([OUTCOME_COL, "myoepithelial_basal_share"]))

    g = share[patient_meta[OUTCOME_COL] == BASELINE_LEVEL]
    p = share[patient_meta[OUTCOME_COL] == COMPARISON_LEVEL]
    auc = mannwhitneyu(g, p).statistic / (len(g) * len(p))
    print(f"{BASELINE_LEVEL}: n={len(g)} median {g.median():.1%}   "
          f"{COMPARISON_LEVEL}: n={len(p)} median {p.median():.1%}")
    print(f"Mann-Whitney p = {mannwhitneyu(p, g).pvalue:.4f}   AUC = {auc:.3f}")

    # cycling rate *within* each lineage: does Poor really proliferate more?
    cyc_lum = None
    if "Actively Dividing cells" in counts and "Actively Dividing Myoepithelial cells - Tumor" in counts:
        cyc_lum = counts["Actively Dividing cells"] / counts[LUM_BUCKET].sum(axis=1)
        cyc_myo = counts["Actively Dividing Myoepithelial cells - Tumor"] / counts[MYO_BUCKET].sum(axis=1)
        print("\nCycling fraction within each lineage:")
        for label, s in [("luminal/ductal", cyc_lum), ("myoepithelial/basal", cyc_myo)]:
            a = s[patient_meta[OUTCOME_COL] == BASELINE_LEVEL]
            b = s[patient_meta[OUTCOME_COL] == COMPARISON_LEVEL]
            print(f"  {label:20s} {BASELINE_LEVEL} {a.median():.1%}  {COMPARISON_LEVEL} {b.median():.1%}"
                  f"   p = {mannwhitneyu(b, a).pvalue:.3f}")
        print("\nIf the luminal cycling rate is flat, the proliferating population is enriched only")
        print("because its lineage expanded - do not report it as increased proliferation.")

    # ---------------------------------------------------------------- figure
    # Panel A is the headline: one number per patient, no model, no reference population.
    # Panel B is the caveat that has to travel with it - the cycling *rate* is flat, so the
    # proliferation effect is a passenger of the lineage shift, not evidence of proliferation.
    # Maxilla patients are drawn as open diamonds because the credible set does not survive
    # their removal (review doc 5.4); the figure should not hide which points carry the result.
    is_maxilla = (patient_meta["site"].astype(str) == "Maxilla").to_numpy()

    npanel = 2 if cyc_lum is not None else 1
    fig_share, axs = plt.subplots(1, npanel, figsize=(4.6*npanel, 4.4), squeeze=False)
    rng = np.random.default_rng(SEED)

    panels = [("A", share, "myoepithelial/basal % of tumour compartment", True)]
    if cyc_lum is not None:
        panels.append(("B", cyc_lum, "% of luminal lineage that is cycling", False))

    for ax, (tag, series, ylab, show_auc) in zip(axs[0], panels):
        vals, jits, mxs = [], [], []
        for j, lvl in enumerate([BASELINE_LEVEL, COMPARISON_LEVEL], start=1):
            sel = (patient_meta[OUTCOME_COL] == lvl).to_numpy()
            v   = series.to_numpy()[sel] * 100
            vals.append(v)
            mxs.append(is_maxilla[sel])
            jits.append(j + rng.uniform(-.10, .10, v.size))

        # boxes first (filled, group colour), points on top in black
        bp = ax.boxplot(vals, labels=[f"{BASELINE_LEVEL}\n(n={vals[0].size})",
                                      f"{COMPARISON_LEVEL}\n(n={vals[1].size})"],
                        showfliers=False, widths=.55, zorder=2, patch_artist=True)
        for box, lvl in zip(bp["boxes"], [BASELINE_LEVEL, COMPARISON_LEVEL]):
            box.set_facecolor(GROUP_COLOURS[lvl])
            box.set_edgecolor(BOX_EDGE)
            box.set_linewidth(1.3)
        for element in ("whiskers", "caps", "medians"):
            for artist in bp[element]:
                artist.set_color(BOX_EDGE)
                artist.set_linewidth(1.3)
        for w in bp["whiskers"]:            # dashed whiskers only; caps/medians stay solid
            w.set_linestyle(WHISKER_STYLE)

        for v, jit, mx in zip(vals, jits, mxs):
            ax.scatter(jit[~mx], v[~mx], s=44, alpha=.9, zorder=3,
                       color=POINT_COLOUR, linewidths=0)
            if mx.any():   # Maxilla: open black diamonds, readable over a filled box
                ax.scatter(jit[mx], v[mx], s=64, zorder=4, marker="D",
                           facecolor="white", edgecolor=POINT_COLOUR, linewidth=1.5)

        pv = mannwhitneyu(vals[1], vals[0]).pvalue
        sub = f"p = {pv:.4f}"
        if show_auc:
            sub += f"   AUC = {mannwhitneyu(vals[0], vals[1]).statistic/(vals[0].size*vals[1].size):.3f}"
        ax.set_title(f"{tag}. {ylab}\n{sub}", fontsize=10)
        ax.set_ylabel("%", fontsize=9)
        ax.set_ylim(-4, 104)
        ax.tick_params(labelsize=9)
        ax.spines[["top", "right"]].set_visible(False)

    if is_maxilla.any():
        axs[0][0].scatter([], [], marker="D", facecolor="white", edgecolor=POINT_COLOUR,
                          linewidth=1.5, s=64, label="Maxilla")
        axs[0][0].legend(fontsize=8, frameon=False, loc="lower left")

    fig_share.tight_layout()
    save_fig(fig_share, f"myoepithelial_share_{CELL_TYPE_COL}")
    plt.show()
else:
    print("skipped (RUN_SHARE_INDEX = False)")

## 12. CSV exports

**Figures are already written** by sections 6 and 11 (controlled by `SAVE_FIGURES`, on by
default) into `outputs/images/`. This section is only for the CSV tables, which stay off unless
you set `SAVE_OUTPUTS = True` in section 1. The `.h5ad` is never written by either switch.

In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stem = f"{CELL_TYPE_COL}_{OUTCOME_COL}"
    counts.to_csv(OUTPUT_DIR / f"{stem}_counts.csv")
    props.to_csv(OUTPUT_DIR / f"{stem}_proportions.csv")
    sparsity.to_csv(OUTPUT_DIR / f"{stem}_sparsity.csv")
    results.to_csv(OUTPUT_DIR / f"{stem}_sccoda_results.csv", index=False)
    proportion_summary.to_csv(OUTPUT_DIR / f"{stem}_proportion_summary.csv", index=False)

    for name, obj in [("reference_sensitivity", "reference_sensitivity"),
                      ("seed_stability", "seed_stability"),
                      ("leave_site_out", "leave_site_out"),
                      ("share_index", "share_table")]:
        if obj in globals():
            globals()[obj].to_csv(OUTPUT_DIR / f"{stem}_{name}.csv")
    print(f"CSVs written to {OUTPUT_DIR}")
else:
    print("SAVE_OUTPUTS = False - no CSVs written (figures are handled separately).")

print(f"\nFigures ({'on' if SAVE_FIGURES else 'OFF'}) -> {IMAGE_DIR}")
for f in sorted(IMAGE_DIR.glob(f"{FIG_PREFIX}_*{CELL_TYPE_COL}.svg")):
    print(f"  {f.name}")

## How to report this

Three populations are credible, and all three are tumour cells: **Myoepithelial cells - Tumor**
relatively depleted in Poor, **Epithelial cells - Tumor** and **Actively Dividing cells**
relatively enriched. No immune, stromal or vascular population shows any outcome association.

Bear in mind when writing:

1. The three effects describe one substitution, not three independent changes. Myoepithelial
   and epithelial tumour content are strongly anti-correlated, and the cycling population is
   luminal-lineage.
2. Whether the cycling rate *within* the luminal lineage differs is a separate question from
   whether the cycling population is larger (section 11).
3. Four Poor-outcome patients have myoepithelial content inside the Good range, so composition
   does not account for every poor outcome.
4. Site and composition cannot be separated in this cohort (section 9).
5. Histologic pattern and grade are not recorded in `clinical_info.csv`, so it cannot be shown
   whether this composition axis adds anything beyond known prognostic histology.